# 🚀 OnMyDisk AI Node (Google Colab + GPU + Gemma 31B / Gemma 27B + ComfyUI)

Этот блокнот разворачивает полноценный **AI-узел OnMyDisk** прямо в среде Google Colab:
1. **GPU-инференс**: Ollama со старшей моделью **Gemma 31B** или **Gemma 27B** (L4 / A100 GPU).
2. **OnMyChat**: Локальный AI-сервис (FastAPI/RAG/Чат) на `localhost:4000` с предустановленной базой знаний OMD.
3. **ComfyUI**: Локальный генератор аватаров и изображений на `localhost:8188` с кастомными узлами и SDXL пайплайном.
4. **OnMyDisk Connector**: Официальный Linux-клиент `onmydisk` с автоматическим туннелированием через шлюз `onmydisk.net`.
5. **Безопасность**: Работает под отдельным настраиваемым системным пользователем (не под `root`).
6. **Google Drive персистентность**: Веса Ollama, модели ComfyUI (`ComfyUI-models`), кеш SentenceTransformers, индекс ChromaDB и SSL-сертификаты ноды сохраняются на Google Диске.
7. **Персистентность и защита от сна**: Демонизация процессов через `setsid` и фоновый Watchdog для работы без открытой вкладки браузера.

### 1. Настройки AI-узла, создание системного пользователя и монтирование Google Drive

In [ ]:
# @title ⚙️ Параметры узла (заполните перед запуском)
NODE_USER = "omduser"  # @param {type:"string"}
NODE_PASSWORD = "colab123"  # @param {type:"string"}
DEVICE_NAME = "Colab-AI-Node"  # @param {type:"string"}
WORK_MODEL = "gemma4:31b"  # @param ["gemma4:31b", "gemma2:27b", "qwen2.5:32b"] {allow-input: true}
FUN_MODEL = "hf.co/mradermacher/gemma-4-31B-it-heretic-GGUF:Q4_K_M"  # @param {type:"string"}
DEFAULT_MODEL = WORK_MODEL
VISION_MODEL = WORK_MODEL
AI_TOKEN = "colab-ai-secret-token"  # @param {type:"string"}

from google.colab import drive
import os, sys, subprocess, time, shutil
if not NODE_PASSWORD:
    NODE_PASSWORD = "colab123"

# 1. Проверяем доступный GPU
!nvidia-smi

# 2. Задаем пароль root и создаем системного пользователя OS виртуалки (не работаем под root)
!echo "root:{NODE_PASSWORD}" | chpasswd
!id -u {NODE_USER} >/dev/null 2>&1 || useradd -m -s /bin/bash -G sudo,video {NODE_USER}
!echo "{NODE_USER}:{NODE_PASSWORD}" | chpasswd
!mkdir -p /home/{NODE_USER}/Data
!chown -R {NODE_USER}:{NODE_USER} /home/{NODE_USER}
print(f"✅ Пользователь '{NODE_USER}' создан с домашней папкой /home/{NODE_USER} и пароль установлен!")

# 3. Монтируем Google Drive для постоянного хранения данных
drive.mount('/content/drive')

NODE_DATA_DIR = "/content/drive/MyDrive/onmydisk-node"
OMD_CONFIG_DIR = f"{NODE_DATA_DIR}/etc_onmydisk"
ONMYCHAT_PERSIST_DIR = f"{NODE_DATA_DIR}/onmychat_data"
UNIFIED_INDEX_DIR = f"{NODE_DATA_DIR}/unified_index"
HF_CACHE_DIR = f"{NODE_DATA_DIR}/huggingface_hub"

for path in [NODE_DATA_DIR, OMD_CONFIG_DIR, ONMYCHAT_PERSIST_DIR, UNIFIED_INDEX_DIR, HF_CACHE_DIR]:
    os.makedirs(path, exist_ok=True)

print("✅ Google Drive смонтирован и папки подготовлены:", NODE_DATA_DIR)

### 2. Установка и запуск Ollama (GPU)

In [ ]:
# 1. Установка zstd и Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Настраиваем быстрый локальный SSD для Ollama (чтение весов в VRAM на скорости ~2 ГБ/с)
LOCAL_OLLAMA = "/root/.ollama/models"
os.makedirs(LOCAL_OLLAMA, exist_ok=True)
os.environ["OLLAMA_MODELS"] = LOCAL_OLLAMA
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# 3. Запускаем Ollama на локальном NVMe диске (отвязываем от сессии через setsid)
print("Запуск службы Ollama...")
ollama_bin = "/usr/local/bin/ollama" if os.path.exists("/usr/local/bin/ollama") else "ollama"
ollama_log = open("/tmp/ollama.log", "a")
ollama_proc = subprocess.Popen([ollama_bin, "serve"], env=os.environ, stdout=ollama_log, stderr=ollama_log, preexec_fn=os.setsid)
time.sleep(3)

# 4. Проверка доступности API
!curl -s http://127.0.0.1:11434/api/version
print("\n✅ Ollama успешно запущена на GPU!")

# 5. Скачивание моделей напрямую из репозиториев
print(f"Проверка/скачивание WORK модели {WORK_MODEL}...")
!ollama pull {WORK_MODEL}
print(f"Проверка/скачивание FUN модели {FUN_MODEL}...")
!ollama pull {FUN_MODEL}
print("✅ Модели готовы к инференсу на полной скорости VRAM!")


### 3. Установка OnMyDisk Connector, зависимостей и симлинков

In [ ]:
# 1. Установка всех необходимых системных зависимостей для Ubuntu 24.04 (Noble)
!apt-get update -qq
!apt-get install -y -qq samba udisks2 libglu1-mesa libtag1v5 ruby-full poppler-utils \
    pandoc build-essential zlib1g-dev ffmpeg socat \
    libqt5core5a libqt5network5 libqt5websockets5 libqt5widgets5 libqt5sql5 libqt5webchannel5

# 2. Совместимость OpenSSL 1.1 и libcrypt.so.2 для бинарников коннектора
!wget -q -nc http://security.ubuntu.com/ubuntu/pool/main/o/openssl/libssl1.1_1.1.1f-1ubuntu2.24_amd64.deb
!dpkg -i libssl1.1_1.1.1f-1ubuntu2.24_amd64.deb || apt-get install -f -y -qq
!ln -sf /lib/x86_64-linux-gnu/libcrypt.so.1 /lib/x86_64-linux-gnu/libcrypt.so.2 2>/dev/null || true
!ln -sf /usr/lib/x86_64-linux-gnu/libcrypt.so.1 /usr/lib/x86_64-linux-gnu/libcrypt.so.2 2>/dev/null || true
!ldconfig

# 3. Скачивание и установка актуального deb-пакета OnMyDisk Connector
DEB_URL = "https://forge.bineon.team/repo/OnMyDisk/Connector/onmydisk-connector-amd64.deb"
!wget -q -O /tmp/onmydisk-connector.deb "{DEB_URL}"
!apt-get install -y -qq /tmp/onmydisk-connector.deb || (dpkg -i /tmp/onmydisk-connector.deb && apt-get install -f -y -qq)

# 4. Восстановление конфигурации и сертификатов с Google Drive (если есть)
!mkdir -p /etc/onmydisk
if os.path.exists(f"{OMD_CONFIG_DIR}/onmydisk.conf"):
    print("Восстанавливаем сохраненную конфигурацию ноды с Google Drive...")
    !cp -ru "{OMD_CONFIG_DIR}"/* /etc/onmydisk/ 2>/dev/null || true
else:
    print("Первый запуск: инициализация новой конфигурации...")

# 5. Настройка пользователя и имени устройства (hostname) в конфигах
for cfg in ["/etc/onmydisk/onmydisk.conf", "/etc/onmydisk/onmydisk.conf.user"]:
    !touch "{cfg}"
    !grep -q "\[FileNode\]" "{cfg}" || echo -e "\n[FileNode]" >> "{cfg}"
    !sed -i "s/^User=.*/User={NODE_USER}/" "{cfg}" 2>/dev/null || true
    !grep -q "User={NODE_USER}" "{cfg}" || sed -i "/\[FileNode\]/a User={NODE_USER}\nGroup={NODE_USER}" "{cfg}"
    !sed -i "s/^Group=.*/Group={NODE_USER}/" "{cfg}" 2>/dev/null || true
    !grep -q "\[Settings\]" "{cfg}" || echo -e "\n[Settings]" >> "{cfg}"
    !sed -i "s/^hostname=.*/hostname={DEVICE_NAME}/" "{cfg}" 2>/dev/null || true
    !grep -q "hostname={DEVICE_NAME}" "{cfg}" || sed -i "/\[Settings\]/a hostname={DEVICE_NAME}" "{cfg}"

!chown -R {NODE_USER}:{NODE_USER} /etc/onmydisk /home/{NODE_USER} /var/onmydisk 2>/dev/null || true

print(f"✅ OnMyDisk Connector настроен для пользователя '{NODE_USER}' с именем '{DEVICE_NAME}'!")

### 4. Установка OnMyChat, архива знаний OMD и модели эмбеддингов

In [ ]:
# 1. Клонируем исходный код OnMyChat
if not os.path.exists("/content/onmychat"):
    !git clone https://github.com/alexeyvolkoff/onmychat.git /content/onmychat
else:
    !cd /content/onmychat && git pull

# 2. Установка зависимостей Python (фиксируем opentelemetry для совместимости с окружением Colab)
!pip install -q "opentelemetry-api<=1.42.1" "opentelemetry-sdk<=1.42.1" fastapi uvicorn requests pillow pydantic chromadb numpy sentence-transformers aiortc

# 3. База знаний OMD (ChromaDB) с персистентностью на Google Drive
os.makedirs("/content/onmychat/memory_index/unified_index", exist_ok=True)
if os.path.exists(f"{UNIFIED_INDEX_DIR}/chroma.sqlite3"):
    print("Восстанавливаем базу знаний OMD из Google Drive...")
    !cp -ru "{UNIFIED_INDEX_DIR}"/* /content/onmychat/memory_index/unified_index/
else:
    print("Скачиваем готовую базу знаний OMD...")
    !wget -q -O /tmp/chroma_omd.tar.gz https://forge.bineon.team/repo/OnMyChat/chroma_omd.tar.gz
    !tar -xzf /tmp/chroma_omd.tar.gz -C /content/onmychat/memory_index/unified_index
    !rm -f /tmp/chroma_omd.tar.gz
    !mkdir -p "{UNIFIED_INDEX_DIR}" && cp -ru /content/onmychat/memory_index/unified_index/* "{UNIFIED_INDEX_DIR}/" 2>/dev/null || true

# 4. Модель эмбеддингов SentenceTransformers с персистентностью на Google Drive
HF_LOCAL_HUB = "/root/.cache/huggingface/hub"
HF_USER_HUB = f"/home/{NODE_USER}/.cache/huggingface/hub"
HF_MODEL_NAME = "models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2"
os.makedirs(HF_LOCAL_HUB, exist_ok=True)
if os.path.exists(f"{HF_CACHE_DIR}/{HF_MODEL_NAME}"):
    print("Восстанавливаем модель эмбеддингов из Google Drive...")
    !cp -ru "{HF_CACHE_DIR}/{HF_MODEL_NAME}" "{HF_LOCAL_HUB}/"
else:
    print("Скачиваем предобученную модель эмбеддингов...")
    !wget -q -O /tmp/sentence_model.tar.gz https://forge.bineon.team/repo/OnMyChat/sentence_model.tar.gz
    !tar -xzf /tmp/sentence_model.tar.gz -C "{HF_LOCAL_HUB}"
    !rm -f /tmp/sentence_model.tar.gz
    !mkdir -p "{HF_CACHE_DIR}" && cp -ru "{HF_LOCAL_HUB}/{HF_MODEL_NAME}" "{HF_CACHE_DIR}/" 2>/dev/null || true

# Копируем кеш модели также в профиль текущего пользователя
!mkdir -p "{HF_USER_HUB}" && cp -ru "{HF_LOCAL_HUB}/{HF_MODEL_NAME}" "{HF_USER_HUB}/" 2>/dev/null || true
!chown -R {NODE_USER}:{NODE_USER} /home/{NODE_USER}/.cache 2>/dev/null || true

# 5. Генерируем полный рабочий config.ini для OnMyChat
config_content = f"""[settings]
DEFAULT_SYSTEM_PROMPT_FILE = default.txt
DEFAULT_ASSISTANT_NAME = June
DEFAULT_ASSISTANT_TITLE = Junior assistant

GATEWAY_URL = https://onmydisk.net
AI_TOKEN = {AI_TOKEN}

OLLAMA_URL = http://127.0.0.1:11434
DEFAULT_MODEL = {WORK_MODEL}
VISION_MODEL = {WORK_MODEL}
WORK_MODEL = {WORK_MODEL}
FUN_MODEL = {FUN_MODEL}
FUN_TEMPERATURE = 0.75
FUN_TOP_P = 0.85
AI_PROVIDER = local

LLM_NUM_CTX = 32768
LLM_NUM_PREDICT = 2048
LLM_KEEP_ALIVE = 300
LLM_STREAM_TIMEOUT = 180
LLM_REQUEST_TIMEOUT = 120

STORAGE_ROOT = /content
COMFY_API_URL = http://127.0.0.1:8188
COMFY_TIMEOUT = 300
WORKFLOW_PATH = flow.json
COMFY_OUTPUT_DIR = /content/ComfyUI/output
COMFY_INPUT_DIR = /content/ComfyUI/input
AVATAR_DIR = avatars
APP_ROOT_DIR = /content/onmychat

HISTORY_LIMIT = 300
HISTORY_DIR = history
SUMMARY_THRESHOLD = 20

REALISTIC_MODEL = CyberRealisticXLPlay_V10.0_FP16.safetensors
PERFECT_MODEL = cyberrealisticPony_v180Coreshift.safetensors
FANTASY_MODEL = CyberIllustrious_V12_FP16.safetensors
TOONED_MODEL = CyberRealistic_PonySemi_V5.safetensors

SEARCH_THRESHOLD = 0.75
RAG_THRESHOLD = 0.75
RAG_RELATIVE_RATIO = 0.65
RAG_TOP_K = 8
SEARCH_TOP_K = 20
TTS_ENABLED = false
"""

with open("/content/onmychat/config.ini", "w") as f:
    f.write(config_content)

!chown -R {NODE_USER}:{NODE_USER} /content/onmychat /home/{NODE_USER}
print(f"✅ OnMyChat настроен с WORK={WORK_MODEL} и FUN={FUN_MODEL} для пользователя {NODE_USER}!")


### 5. Установка и запуск ComfyUI (генерация аватаров и изображений)

Для работы генерации аватаров и изображений загрузите чекпоинты и LoRA на Google Диск в папку:
`/content/drive/MyDrive/ComfyUI-models/checkpoints/` и `/content/drive/MyDrive/ComfyUI-models/loras/`

In [ ]:
# 1. Клонируем ComfyUI и устанавливаем зависимости
if not os.path.exists("/content/ComfyUI"):
    print("Установка ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
    !cd /content/ComfyUI && pip install -q -r requirements.txt

# 2. Устанавливаем необходимые Custom Nodes (rgthree и comfyui_essentials для flow.json)
CUSTOM_NODES = "/content/ComfyUI/custom_nodes"
os.makedirs(CUSTOM_NODES, exist_ok=True)

if not os.path.exists(f"{CUSTOM_NODES}/rgthree-comfy"):
    print("Установка кастомного узла rgthree-comfy (Power Lora Loader)...")
    !git clone https://github.com/rgthree/rgthree-comfy {CUSTOM_NODES}/rgthree-comfy
    !pip install -q -r {CUSTOM_NODES}/rgthree-comfy/requirements.txt 2>/dev/null || true

if not os.path.exists(f"{CUSTOM_NODES}/comfyui_essentials"):
    print("Установка кастомного узла ComfyUI_essentials (ImageCASharpening)...")
    !git clone https://github.com/cubiq/ComfyUI_essentials {CUSTOM_NODES}/comfyui_essentials
    !pip install -q -r {CUSTOM_NODES}/comfyui_essentials/requirements.txt 2>/dev/null || true

# 3. Подключаем модели с Google Диска (папка ComfyUI-models)
COMFY_DRIVE_DIR = "/content/drive/MyDrive/ComfyUI-models"
os.makedirs(f"{COMFY_DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{COMFY_DRIVE_DIR}/loras", exist_ok=True)
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)
os.makedirs("/content/ComfyUI/output", exist_ok=True)
os.makedirs("/content/ComfyUI/input", exist_ok=True)
!chown -R {NODE_USER}:{NODE_USER} /content/ComfyUI/output /content/ComfyUI/input 2>/dev/null || true
!chmod 777 /content/ComfyUI/output /content/ComfyUI/input 2>/dev/null || true

# Симлинкуем чекпоинты и LoRA из Google Drive в ComfyUI
!cp -ru "{COMFY_DRIVE_DIR}/checkpoints"/* /content/ComfyUI/models/checkpoints/ 2>/dev/null || true
!ln -sf "{COMFY_DRIVE_DIR}/loras"/* /content/ComfyUI/models/loras/ 2>/dev/null || true
# Если файлы залиты прямо в корень ComfyUI-models — подхватываем их тоже
!find "{COMFY_DRIVE_DIR}" -maxdepth 1 -name "*.safetensors" -exec ln -sf {} /content/ComfyUI/models/checkpoints/ \; 2>/dev/null || true

# Выводим список обнаруженных моделей
print("Обнаруженные чекпоинты:")
!ls -lh /content/ComfyUI/models/checkpoints/ || echo "Пока нет чекпоинтов"
print("\nОбнаруженные LoRA:")
!ls -lh /content/ComfyUI/models/loras/ || echo "Пока нет LoRA"

# 4. Запуск службы ComfyUI в фоне (порт 8188)
!pkill -f "main.py.*8188" 2>/dev/null || true
comfy_log = open("/tmp/comfyui.log", "a")
comfy_proc = subprocess.Popen(["python3", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--preview-method", "auto"], cwd="/content/ComfyUI", stdout=comfy_log, stderr=comfy_log, preexec_fn=os.setsid)

print("\nОжидание запуска ComfyUI API (порт 8188)...\n")
import urllib.request, time
comfy_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=1) as resp:
            if resp.status == 200:
                comfy_ready = True
                break
    except Exception:
        time.sleep(1)

if comfy_ready:
    print("✅ ComfyUI успешно запущен и готов к генерации на http://127.0.0.1:8188!")
else:
    print("⚠️ ComfyUI запускается, проверьте /tmp/comfyui.log при необходимости.")


### 6. Персистентный запуск OnMyChat и OnMyDisk Connector

In [ ]:
# 1. Останавливаем старые зависшие процессы и очищаем логи
!pkill -f "api.py" 2>/dev/null || true
!killall onmydisk 2>/dev/null || true
!killall uvicorn 2>/dev/null || true
!rm -f /tmp/onmychat.log /tmp/onmydisk.log
!touch /tmp/onmychat.log /tmp/onmydisk.log
!chown {NODE_USER}:{NODE_USER} /tmp/onmychat.log /tmp/onmydisk.log
!chown -R {NODE_USER}:{NODE_USER} /content/onmychat /content/ComfyUI/output /content/ComfyUI/input 2>/dev/null || true

# 2. Запускаем OnMyChat (порт 4000) и OnMyDisk Connector с небуферизованным выводом
chat_env = os.environ.copy()
chat_env["PYTHONUNBUFFERED"] = "1"
chat_log = open("/tmp/onmychat.log", "a")
chat_env["HOME"] = f"/home/{NODE_USER}"
chat_env["USER"] = NODE_USER
chat_env["HF_HOME"] = f"/home/{NODE_USER}/.cache/huggingface"
# Запуск OnMyChat от имени системного пользователя NODE_USER (не root)
!su - {NODE_USER} -c "cd /content/onmychat && PYTHONUNBUFFERED=1 HF_HOME=/home/{NODE_USER}/.cache/huggingface nohup python3 -u api.py 4000 </dev/null >>/tmp/onmychat.log 2>&1 &"

omd_log = open("/tmp/onmydisk.log", "a")
omd_proc = subprocess.Popen(["/usr/bin/onmydisk"], stdout=omd_log, stderr=omd_log, preexec_fn=os.setsid)

# 3. Ожидание инициализации OnMyChat API (с проверкой готовности)
print("Ожидание старта сервисов...")
import urllib.request, re

ready = False
for attempt in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:4000/docs", timeout=1) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if ready:
    print("✅ OnMyChat API готов на http://127.0.0.1:4000")
else:
    print("⚠️ OnMyChat еще инициализируется, проверьте логи ниже")

# 4. Сохраняем сгенерированные сертификаты и конфиги на Google Drive
!cp -ru /etc/onmydisk/* "{OMD_CONFIG_DIR}/" 2>/dev/null || true

# 5. Извлекаем Device ID из лога подключения OnMyDisk
time.sleep(2)
device_id = None
if os.path.exists("/tmp/onmydisk.log"):
    with open("/tmp/onmydisk.log", "r", errors="ignore") as f:
        for line in f:
            m = re.search(r'Certificate loaded for\s+"([^"]+)"', line)
            if m:
                device_id = m.group(1)
                break

print("\n--- Лог OnMyChat ---")
!tail -n 10 /tmp/onmychat.log

print("\n--- Лог OnMyDisk Connector ---")
!tail -n 10 /tmp/onmydisk.log

# 6. Инструкция для подключения в браузере
print("\n" + "="*72)
print("🎉 Узел OnMyDisk успешно запущен!")
print("="*72)
print("1. Откройте в браузере: https://direct.onmydisk.net")
print("2. Войдите в свой аккаунт OnMyDisk и нажмите «Подключить устройство»:")
print(f"   • ID устройства:   {device_id if device_id else 'см. лог выше'}")
print(f"   • Имя устройства:  {DEVICE_NAME}")
print(f"   • Пользователь:    {NODE_USER}")
print(f"   • Пароль:          {NODE_PASSWORD}")
print("="*72 + "\n")

### 7. Мониторинг, автоперезапуск (Watchdog) и защита от сна

**Как сделать работу сервисов персистентной при закрытии вкладки браузера:**
1. **Включите фоновое выполнение (Google Colab Pro)**: Меню *Среда выполнения (Runtime)* → *Изменить тип среды выполнения* → включите галочку **Фоновое выполнение (Run in background)**.
2. **Запустите ячейку ниже**: Она работает как лёгкий сторожевой таймер (Watchdog), предотвращая засыпание среды, отслеживая состояние процессов `ollama`, `comfyui`, `onmychat` и `onmydisk`, и автоматически перезапуская их при падении.
3. **Google Drive персистентность**: Все сертификаты, аккаунт, веса моделей и база знаний сохраняются на Google Диске. Повторный запуск восстанавливает ноду за секунды.

In [ ]:
# Фоновый Watchdog и монитор здоровья всех сервисов (Ollama + ComfyUI + OnMyChat + OnMyDisk)
import time, subprocess, os

print("🛡️ Запуск Watchdog мониторинга (Ctrl+C или 'Прервать выполнение' для остановки)...\n")

try:
    while True:
        # 1. Проверяем Ollama
        ollama_check = subprocess.run(["pgrep", "-f", "ollama serve"], capture_output=True)
        if ollama_check.returncode != 0:
            print(f"[{time.strftime('%X')}] ⚠️ Ollama не отвечает, перезапуск...")
            ollama_log = open("/tmp/ollama.log", "a")
            subprocess.Popen([ollama_bin, "serve"], env=os.environ, stdout=ollama_log, stderr=ollama_log, preexec_fn=os.setsid)

        # 2. Проверяем ComfyUI
        comfy_check = subprocess.run(["pgrep", "-f", "main.py.*8188"], capture_output=True)
        if comfy_check.returncode != 0:
            print(f"[{time.strftime('%X')}] ⚠️ ComfyUI упал, перезапуск...")
            comfy_log = open("/tmp/comfyui.log", "a")
            subprocess.Popen(["python3", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--preview-method", "auto"], cwd="/content/ComfyUI", stdout=comfy_log, stderr=comfy_log, preexec_fn=os.setsid)

        # 3. Проверяем OnMyChat
        chat_check = subprocess.run(["pgrep", "-f", "api.py"], capture_output=True)
        if chat_check.returncode != 0:
            print(f"[{time.strftime('%X')}] ⚠️ OnMyChat упал, перезапуск...")
            chat_log = open("/tmp/onmychat.log", "a")
            chat_env = os.environ.copy()
            chat_env["PYTHONUNBUFFERED"] = "1"
            chat_env["HOME"] = f"/home/{NODE_USER}"
            chat_env["USER"] = NODE_USER
            chat_env["HF_HOME"] = f"/home/{NODE_USER}/.cache/huggingface"
            subprocess.run(f"su - {NODE_USER} -c "cd /content/onmychat && PYTHONUNBUFFERED=1 HF_HOME=/home/{NODE_USER}/.cache/huggingface nohup python3 -u api.py 4000 </dev/null >>/tmp/onmychat.log 2>&1 &"", shell=True)

        # 4. Проверяем OnMyDisk
        omd_check = subprocess.run(["pgrep", "-x", "onmydisk"], capture_output=True)
        if omd_check.returncode != 0:
            print(f"[{time.strftime('%X')}] ⚠️ OnMyDisk упал, перезапуск...")
            omd_log = open("/tmp/onmydisk.log", "a")
            subprocess.Popen(["/usr/bin/onmydisk"], stdout=omd_log, stderr=omd_log, preexec_fn=os.setsid)

        # 5. Синхронизируем конфиги и сертификаты ноды на Google Drive
        subprocess.run(f"cp -ru /etc/onmydisk/* \"{OMD_CONFIG_DIR}/\" 2>/dev/null || true", shell=True)

        time.sleep(60)
except KeyboardInterrupt:
    print("\nWatchdog остановлен.")
